# Telecom Customer Churn — Retention Intelligence

## Objective

The objective of this notebook is to transform the finalized churn-risk,
behavioural, customer-value and prioritization outputs into a unified
retention-intelligence layer.

This layer connects predictive churn risk with observable behavioural
deterioration, relative customer value, retention priority and candidate
retention-review actions.

The purpose is not to train another machine-learning model. Instead, this
notebook converts the outputs of the completed modelling stage into a
customer-level decision dataset that can be consumed by the final dashboard
and AI retention agent.

The analysis will establish:

* a unified customer-level retention record,
* risk, behaviour and value segmentation,
* retention priority,
* candidate action routing,
* supporting behavioural evidence,
* and a final decision-ready dataset.

The resulting dataset will serve as the interface between the analytical
pipeline and the operational dashboard.


In [1]:
# ============================================================
# Retention Intelligence
# Core dependencies
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path 

In [3]:
# ============================================================
# Project paths
# ============================================================

PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telecom_churn_prepared.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

print(f"Prepared dataset : {DATA_PATH.resolve()}")
print(f"Output directory : {OUTPUT_DIR.resolve()}")

Prepared dataset : E:\Churn Prediction\data\processed\telecom_churn_prepared.csv
Output directory : E:\Churn Prediction\data\processed


In [4]:
# ============================================================
# Load the prepared customer dataset.
# ============================================================

df_prepared = pd.read_csv(DATA_PATH)

print("Prepared dataset loaded successfully.")
print(f"Rows    : {df_prepared.shape[0]}")
print(f"Columns : {df_prepared.shape[1]}")

Prepared dataset loaded successfully.
Rows    : 69999
Columns : 191


In [5]:
# ============================================================
# Validate the prepared dataset before constructing the
# retention-intelligence layer.
# ============================================================

assert "id" in df_prepared.columns
assert "churn_probability" in df_prepared.columns

assert df_prepared["id"].notna().all()
assert df_prepared["id"].is_unique

print("Dataset validation passed.")
print(f"Unique customers : {df_prepared['id'].nunique():,}")

Dataset validation passed.
Unique customers : 69,999


In [6]:
# ============================================================
# Load the frozen churn-risk model.
# No model training is performed in this notebook.
# ============================================================

import joblib

MODEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final_lightgbm_churn_model.joblib"
)

lgbm_model = joblib.load(MODEL_PATH)

print("Frozen LightGBM model loaded successfully.")
print(f"Model path: {MODEL_PATH.resolve()}")

Frozen LightGBM model loaded successfully.
Model path: E:\Churn Prediction\data\processed\final_lightgbm_churn_model.joblib


In [7]:
# ============================================================
# Generate churn probabilities using the frozen model.
# ============================================================

TARGET = "churn_probability"
CUSTOMER_ID = "id"

X_full = df_prepared.drop(
    columns=[TARGET, CUSTOMER_ID]
)

predicted_risk = lgbm_model.predict_proba(
    X_full
)[:, 1]

risk_output = pd.DataFrame(
    {
        "id": df_prepared[CUSTOMER_ID].values,
        "actual_churn": df_prepared[TARGET].values,
        "predicted_churn_risk": predicted_risk
    },
    index=df_prepared.index
)

print("Risk scores generated.")
print(f"Customers scored : {len(risk_output):,}")
print(
    f"Risk range       : "
    f"{risk_output['predicted_churn_risk'].min():.4f} - "
    f"{risk_output['predicted_churn_risk'].max():.4f}"
)

Risk scores generated.
Customers scored : 69,999
Risk range       : 0.0001 - 0.9969


In [8]:
# ============================================================
# Reconstruct the evaluation population used during
# model development.
# ============================================================

from sklearn.model_selection import train_test_split

TARGET = "churn_probability"
CUSTOMER_ID = "id"

X_full = df_prepared.drop(
    columns=[TARGET, CUSTOMER_ID]
)

y_full = df_prepared[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42
)

validation_index = X_valid.index

retention_df = df_prepared.loc[
    validation_index
].copy()

print("Evaluation population reconstructed successfully.")
print(f"Customers in retention layer : {len(retention_df):,}")
print(
    f"Observed churners             : "
    f"{retention_df[TARGET].sum():,.0f}"
)
print(
    f"Observed churn rate           : "
    f"{retention_df[TARGET].mean():.2%}"
)

Evaluation population reconstructed successfully.
Customers in retention layer : 14,000
Observed churners             : 1,426
Observed churn rate           : 10.19%


In [9]:
# ============================================================
# Risk Segmentation
# Converts continuous churn probabilities into
# relative risk tiers for retention analysis.
# ============================================================

risk_scores = lgbm_model.predict_proba(
    X_valid
)[:, 1]

retention_df["predicted_churn_risk"] = risk_scores

retention_df["risk_tier"] = pd.qcut(
    retention_df["predicted_churn_risk"],
    q=4,
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

risk_summary = (
    retention_df
    .groupby("risk_tier", observed=False)
    .agg(
        Customers=("id", "count"),
        Actual_Churners=(TARGET, "sum"),
        Churn_Rate=(TARGET, "mean"),
        Average_Risk=("predicted_churn_risk", "mean")
    )
    .reset_index()
)

risk_summary["Churn_Rate"] = (
    risk_summary["Churn_Rate"] * 100
)

risk_summary

,risk_tier,Customers,Actual_Churners,Churn_Rate,Average_Risk
0,Low,3500,13,0.371429,0.001785
1,Moderate,3500,29,0.828571,0.005639
2,High,3500,82,2.342857,0.017160
3,Very High,3500,1302,37.200000,0.352869


In [10]:
# ============================================================
# Behavioural State Classification
#
# Purpose:
# Converts the engineered temporal deterioration indicators
# into a single interpretable behavioural state for each
# customer in the validation population.
#
# The rules are intentionally kept identical to the definitions
# established during model development so that the retention
# intelligence layer does not introduce a new analytical rule.
# ============================================================


def classify_behavioral_state(row):
    # Extract the three behavioural dimensions used by the
    # classification logic for the current customer.
    recent = row["recent_deterioration_count"]
    persistent = row["persistent_deterioration_count"]
    coordinated = row["coordinated_deterioration_count"]

    # Identify customers showing both sustained and broad
    # deterioration across the observed monthly period.
    if persistent >= 4 and coordinated >= 4:
        return "Severe Persistent Deterioration"

    # Identify customers whose deterioration has persisted
    # across multiple behavioural dimensions.
    elif persistent >= 2:
        return "Persistent Deterioration"

    # Identify customers with broad deterioration concentrated
    # in the most recent observed period.
    elif recent >= 4 and coordinated >= 4:
        return "Recent Broad Deterioration"

    # Identify customers showing meaningful recent deterioration
    # even when it is not broad enough to qualify as a severe
    # or persistent pattern.
    elif recent >= 2:
        return "Recent Deterioration"

    # Customers not satisfying any deterioration rule are kept
    # as the baseline behavioural group.
    else:
        return "Stable / Limited Deterioration"


# Apply the frozen behavioural classification to every customer
# in the validation population used by the retention layer.
retention_df["behavioral_state"] = retention_df.apply(
    classify_behavioral_state,
    axis=1
)

print("Behavioural states assigned successfully.")
print(
    f"Customers classified : "
    f"{retention_df['behavioral_state'].notna().sum():,}"
)

Behavioural states assigned successfully.
Customers classified : 14,000


In [11]:
# ============================================================
# Behavioural State Summary
#
# Purpose:
# Quantifies the observed churn behaviour and model risk
# associated with each behavioural state.
#
# This is an analytical validation of the state definitions,
# not a claim that the behavioural state causes churn.
# ============================================================


behavior_summary = (
    retention_df
    .groupby("behavioral_state")
    .agg(
        # Number of validation customers assigned to the state.
        Customers=("id", "count"),
           
        # Number of customers in the state who actually churned.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn rate within the behavioural state.
        Churn_Rate=(TARGET, "mean"),

        # Average model-predicted churn probability for the state.
        Average_Risk=("predicted_churn_risk", "mean")
    )
    .reset_index()
)


# Convert the observed churn rate from a proportion to a
# percentage so the output is easier to interpret.
behavior_summary["Churn_Rate"] = (
    behavior_summary["Churn_Rate"] * 100
)


# Sort the states by observed churn rate so that the
# behavioural-risk progression is immediately visible.
behavior_summary = behavior_summary.sort_values(
    by="Churn_Rate",
    ascending=False
).reset_index(drop=True)


behavior_summary

,behavioral_state,Customers,Actual_Churners,Churn_Rate,Average_Risk
0,Severe Persistent Deterioration,1043,281,26.941515,0.246845
1,Recent Broad Deterioration,843,146,17.319098,0.155870
2,Persistent Deterioration,2381,303,12.725745,0.117246
3,Stable / Limited Deterioration,5260,433,8.231939,0.079973
4,Recent Deterioration,4473,263,5.879723,0.051958


In [12]:
# ============================================================
# Risk × Behaviour Intelligence
#
# Purpose:
# Combines model-derived risk segmentation with the
# behavioural state assigned from observed customer
# deterioration patterns.
#
# This creates the first customer segmentation layer that
# considers both predicted churn risk and behavioural context.
# ============================================================


risk_behavior_summary = (
    retention_df
    .groupby(
        ["risk_tier", "behavioral_state"],
        observed=False
    )
    .agg(
        # Number of validation customers belonging to this
        # particular risk-behaviour combination.
        Customers=("id", "count"),

        # Number of customers in the combination who actually
        # churned in the observed target data.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn frequency within the combination.
        Churn_Rate=(TARGET, "mean"),

        # Average probability assigned by the frozen churn model.
        Average_Risk=("predicted_churn_risk", "mean")
    )
    .reset_index()
)


# Convert churn rate from a proportion into percentage form
# for easier interpretation in reports and dashboards.
risk_behavior_summary["Churn_Rate"] = (
    risk_behavior_summary["Churn_Rate"] * 100
)


# Sort first by risk tier and then by behavioural state so
# that the resulting table follows the analytical hierarchy.
risk_order = [
    "Low",
    "Moderate",
    "High",
    "Very High"
]

behavior_order = [
    "Severe Persistent Deterioration",
    "Recent Broad Deterioration",
    "Persistent Deterioration",
    "Recent Deterioration",
    "Stable / Limited Deterioration"
]

risk_behavior_summary["risk_tier"] = pd.Categorical(
    risk_behavior_summary["risk_tier"],
    categories=risk_order,
    ordered=True
)

risk_behavior_summary["behavioral_state"] = pd.Categorical(
    risk_behavior_summary["behavioral_state"],
    categories=behavior_order,
    ordered=True
)


# Apply the defined analytical ordering before displaying
# the final Risk × Behaviour segmentation.
risk_behavior_summary = (
    risk_behavior_summary
    .sort_values(
        ["risk_tier", "behavioral_state"]
    )
    .reset_index(drop=True)
)


risk_behavior_summary

,risk_tier,behavioral_state,Customers,Actual_Churners,Churn_Rate,Average_Risk
0,Low,Severe Persistent Deterioration,68,1,1.470588,0.001930
1,Low,Recent Broad Deterioration,85,1,1.176471,0.002034
2,Low,Persistent Deterioration,466,0,0.000000,0.001821
3,Low,Recent Deterioration,1280,4,0.312500,0.001768
4,Low,Stable / Limited Deterioration,1601,7,0.437227,0.001770
5,Moderate,Severe Persistent Deterioration,146,2,1.369863,0.005858
6,Moderate,Recent Broad Deterioration,142,0,0.000000,0.005617
7,Moderate,Persistent Deterioration,579,7,1.208981,0.005619
8,Moderate,Recent Deterioration,1203,8,0.665004,0.005653
9,Moderate,Stable / Limited Deterioration,1430,12,0.839161,0.005614


In [13]:
# ============================================================
# Customer Value Segmentation
#
# Purpose:
# Represents recent customer value using the latest observed
# ARPU and converts it into relative value tiers.
#
# ARPU is used as a revenue proxy rather than a complete
# customer-lifetime-value or profitability measure.
# ============================================================


VALUE_FEATURE = "arpu_8"


# Verify that the latest monthly ARPU required for value
# segmentation is available for every validation customer.
assert retention_df[VALUE_FEATURE].notna().all(), (
    "Missing ARPU values detected in the validation population."
)


# Divide customers into four equal-frequency groups based on
# their latest observed ARPU.
#
# q=4 creates quartiles rather than fixed monetary thresholds,
# making the segmentation relative to this customer population.
retention_df["value_tier"] = pd.qcut(
    retention_df[VALUE_FEATURE],
    q=4,
    labels=[
        "Lower",
        "Moderate",
        "Higher",
        "Highest"
    ]
)


print("Customer value tiers assigned successfully.")
print(
    f"Customers segmented : "
    f"{retention_df['value_tier'].notna().sum():,}"
)

Customer value tiers assigned successfully.
Customers segmented : 14,000


In [14]:
# ============================================================
# Value Tier Summary
#
# Purpose:
# Examines customer distribution, observed churn and ARPU
# across the four relative value segments.
# ============================================================


value_summary = (
    retention_df
    .groupby("value_tier", observed=False)
    .agg(
        # Number of validation customers assigned to the
        # corresponding relative-value segment.
        Customers=("id", "count"),

        # Number of customers in the segment who actually churned.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn frequency within the value segment.
        Churn_Rate=(TARGET, "mean"),

        # Average latest-month ARPU for the segment.
        Average_ARPU=(VALUE_FEATURE, "mean"),

        # Total observed August ARPU across customers in the
        # segment. This represents revenue exposure, not revenue lost.
        Total_ARPU=(VALUE_FEATURE, "sum")
    )
    .reset_index()
)


# Express observed churn rate as a percentage for
# easier interpretation in reports and dashboards.
value_summary["Churn_Rate"] = (
    value_summary["Churn_Rate"] * 100
)


value_summary

,value_tier,Customers,Actual_Churners,Churn_Rate,Average_ARPU,Total_ARPU
0,Lower,3500,932,26.628571,34.151736,119531.075
1,Moderate,3500,213,6.085714,139.280239,487480.838
2,Higher,3500,156,4.457143,273.027601,955596.605
3,Highest,3500,125,3.571429,685.715875,2400005.564


In [15]:
# ============================================================
# Risk × Value Intelligence
#
# Purpose:
# Combines predicted churn risk with relative customer value
# to identify segments with different levels of potential
# retention exposure.
#
# The analysis does not assume that high-value customers are
# automatically more likely to churn or more profitable to retain.
# ============================================================


risk_value_summary = (
    retention_df
    .groupby(
        ["risk_tier", "value_tier"],
        observed=False
    )
    .agg(
        # Number of customers in each risk-value combination.
        Customers=("id", "count"),

        # Number of observed churners in the combination.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn rate for the combination.
        Churn_Rate=(TARGET, "mean"),

        # Average predicted churn probability.
        Average_Risk=("predicted_churn_risk", "mean"),

        # Average latest observed ARPU.
        Average_ARPU=(VALUE_FEATURE, "mean"),

        # Aggregate observed ARPU representing the revenue
        # exposure within this segment.
        Total_ARPU=(VALUE_FEATURE, "sum")
    )
    .reset_index()
)


# Convert the observed churn rate into percentage form.
risk_value_summary["Churn_Rate"] = (
    risk_value_summary["Churn_Rate"] * 100
)


# Preserve the same business-friendly ordering used
# throughout the retention-intelligence pipeline.
risk_order = [
    "Low",
    "Moderate",
    "High",
    "Very High"
]

value_order = [
    "Lower",
    "Moderate",
    "Higher",
    "Highest"
]


risk_value_summary["risk_tier"] = pd.Categorical(
    risk_value_summary["risk_tier"],
    categories=risk_order,
    ordered=True
)

risk_value_summary["value_tier"] = pd.Categorical(
    risk_value_summary["value_tier"],
    categories=value_order,
    ordered=True
)


risk_value_summary = (
    risk_value_summary
    .sort_values(
        ["risk_tier", "value_tier"]
    )
    .reset_index(drop=True)
)


risk_value_summary

,risk_tier,value_tier,Customers,Actual_Churners,Churn_Rate,Average_Risk,Average_ARPU,Total_ARPU
0,Low,Lower,247,0,0.000000,0.002092,57.014130,14082.490
1,Low,Moderate,767,4,0.521512,0.001913,145.768402,111804.364
2,Low,Higher,1117,2,0.179051,0.001778,273.837507,305876.495
3,Low,Highest,1369,7,0.511322,0.001665,695.987144,952806.400
4,Moderate,Lower,545,7,1.284404,0.005895,54.100490,29484.767
5,Moderate,Moderate,1047,6,0.573066,0.005613,139.143780,145683.538
6,Moderate,Higher,967,8,0.827301,0.005596,273.231086,264214.460
7,Moderate,Highest,941,8,0.850159,0.005562,657.409132,618621.993
8,High,Lower,866,21,2.424942,0.017807,47.302637,40964.084
9,High,Moderate,1017,31,3.048181,0.016910,136.888287,139215.388


In [16]:
# ============================================================
# Retention Priority Assignment
#
# Purpose:
# Converts the validated Risk × Behaviour × Value segmentation
# into an operational retention-priority classification.
#
# Priority is designed for review ordering rather than as a
# prediction target. The rules combine model risk, relative
# customer value and observable behavioural deterioration.
# ============================================================


# Start every customer in the lowest-priority review group.
# Higher-priority rules are then applied explicitly below.
retention_df["retention_priority"] = "Priority 3"


# ============================================================
# Priority 1
#
# Customers with Very High predicted churn risk and relatively
# high observed ARPU are placed first because they combine
# elevated churn risk with greater revenue exposure.
# ============================================================

priority_1_condition = (
    (retention_df["risk_tier"] == "Very High")
    &
    (retention_df["value_tier"].isin([
        "Higher",
        "Highest"
    ]))
)


retention_df.loc[
    priority_1_condition,
    "retention_priority"
] = "Priority 1"


# ============================================================
# Priority 2
#
# Customers with Very High churn risk but lower/moderate value
# are still important when their behaviour shows persistent
# or broad deterioration.
# ============================================================

priority_2_condition = (
    (retention_df["risk_tier"] == "Very High")
    &
    (retention_df["value_tier"].isin([
        "Lower",
        "Moderate"
    ]))
    &
    (retention_df["behavioral_state"].isin([
        "Severe Persistent Deterioration",
        "Persistent Deterioration",
        "Recent Broad Deterioration"
    ]))
)


retention_df.loc[
    priority_2_condition,
    "retention_priority"
] = "Priority 2"


print("Retention priorities assigned successfully.")
print(
    f"Customers prioritized : "
    f"{retention_df['retention_priority'].notna().sum():,}"
)

Retention priorities assigned successfully.
Customers prioritized : 14,000


In [17]:
# ============================================================
# Retention Priority Summary
#
# Purpose:
# Measures the size, observed churn and model risk of each
# retention-priority group.
#
# This validates whether the prioritization logic concentrates
# customers with greater observed churn risk.
# ============================================================


priority_summary = (
    retention_df
    .groupby("retention_priority")
    .agg(
        # Number of validation customers assigned to the priority.
        Customers=("id", "count"),

        # Number of customers who actually churned.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn rate within the priority group.
        Churn_Rate=(TARGET, "mean"),

        # Average predicted churn probability.
        Average_Risk=("predicted_churn_risk", "mean"),

        # Average latest observed ARPU.
        Average_ARPU=(VALUE_FEATURE, "mean"),

        # Aggregate observed ARPU representing revenue exposure.
        Total_ARPU=(VALUE_FEATURE, "sum")
    )
    .reset_index()
)


# Convert churn rate into percentage form for reporting.
priority_summary["Churn_Rate"] = (
    priority_summary["Churn_Rate"] * 100
)


# Preserve the intended operational ordering.
priority_order = [
    "Priority 1",
    "Priority 2",
    "Priority 3"
]


priority_summary["retention_priority"] = pd.Categorical(
    priority_summary["retention_priority"],
    categories=priority_order,
    ordered=True
)


priority_summary = (
    priority_summary
    .sort_values("retention_priority")
    .reset_index(drop=True)
)


priority_summary

,retention_priority,Customers,Actual_Churners,Churn_Rate,Average_Risk,Average_ARPU,Total_ARPU
0,Priority 1,989,226,22.851365,0.200338,474.832893,469609.731
1,Priority 2,1365,599,43.882784,0.419203,40.511327,55297.962
2,Priority 3,11646,601,5.160570,0.047290,295.183444,3437706.389


In [18]:
# ============================================================
# Candidate Action Routing
#
# Purpose:
# Maps observable customer deterioration patterns to a
# primary retention-review category.
#
# The routing is intentionally rule-based and interpretable.
# It does not estimate treatment effectiveness or causal uplift.
# It identifies what should be investigated by the retention team.
# ============================================================


# Initialise every customer with a general retention review.
# This also provides a fallback when no specific signal is
# strong enough to justify a specialised review category.
retention_df["candidate_action"] = (
    "General Retention Review"
)


# ============================================================
# Recharge / Affordability Signal
#
# A decline in either recharge amount or recharge frequency
# can indicate reduced recharge activity and should trigger
# an affordability/recharge review when no stronger behavioural
# routing has already been assigned.
# ============================================================

recharge_signal = (
    (
        df_prepared.loc[
            retention_df.index,
            "recharge_amount_jul_aug_change"
        ] < 0
    )
    |
    (
        df_prepared.loc[
            retention_df.index,
            "recharge_count_jul_aug_change"
        ] < 0
    )
)


# ============================================================
# Usage / Engagement Signal
#
# A decline in outgoing or incoming usage is treated as an
# engagement-related signal when stronger deterioration
# categories do not already apply.
# ============================================================

usage_signal = (
    (
        df_prepared.loc[
            retention_df.index,
            "outgoing_mou_jul_aug_change"
        ] < 0
    )
    |
    (
        df_prepared.loc[
            retention_df.index,
            "incoming_mou_jul_aug_change"
        ] < 0
    )
)


# ============================================================
# Strong Persistent Behaviour
#
# Persistent deterioration receives the highest behavioural
# routing precedence because it represents deterioration
# across the observed monthly period rather than a single
# recent change.
# ============================================================

retention_df.loc[
    retention_df["behavioral_state"].isin([
        "Severe Persistent Deterioration",
        "Persistent Deterioration"
    ]),
    "candidate_action"
] = "Persistent Deterioration Review"


# ============================================================
# Broad Recent Behaviour
#
# Customers showing broad deterioration concentrated in the
# latest observed period receive a dedicated recent-deterioration
# review category.
# ============================================================

retention_df.loc[
    retention_df["behavioral_state"] == (
        "Recent Broad Deterioration"
    ),
    "candidate_action"
] = "Broad Recent Deterioration Review"


# ============================================================
# Recharge Routing
#
# Recharge-related routing is applied only where a stronger
# behavioural state has not already assigned an action.
# ============================================================

retention_df.loc[
    (
        retention_df["candidate_action"]
        == "General Retention Review"
    )
    & recharge_signal,
    "candidate_action"
] = "Recharge / Affordability Review"


# ============================================================
# Usage Routing
#
# Usage-related routing is applied only to customers who have
# not already been assigned a stronger behavioural or recharge
# review category.
# ============================================================

retention_df.loc[
    (
        retention_df["candidate_action"]
        == "General Retention Review"
    )
    & usage_signal,
    "candidate_action"
] = "Usage / Engagement Review"


print("Candidate action routing completed.")
print(
    f"Customers routed : "
    f"{retention_df['candidate_action'].notna().sum():,}"
)

Candidate action routing completed.
Customers routed : 14,000


In [20]:
# ============================================================
# Candidate Action Summary
#
# Purpose:
# Evaluates the distribution, observed churn and predicted
# risk associated with each candidate retention-review category.
#
# The results are descriptive and must not be interpreted as
# evidence that any candidate action prevents churn.
# ============================================================


action_summary = (
    retention_df
    .groupby("candidate_action")
    .agg(
        # Number of validation customers assigned to the
        # corresponding review category.
        Customers=("id", "count"),

        # Number of customers in the category who actually churned.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn rate within the action category.
        Churn_Rate=(TARGET, "mean"),

        # Average predicted churn probability.
        Average_Risk=("predicted_churn_risk", "mean"),

        # Average latest observed customer revenue proxy.
        Average_ARPU=(VALUE_FEATURE, "mean")
    )
    .reset_index()
)


# Convert observed churn rate into percentage form.
action_summary["Churn_Rate"] = (
    action_summary["Churn_Rate"] * 100
)


# Sort categories by observed churn rate so the strongest
# observed-risk groups appear first.
action_summary = (
    action_summary
    .sort_values(
        "Churn_Rate",
        ascending=False
    )
    .reset_index(drop=True)
)


action_summary

,candidate_action,Customers,Actual_Churners,Churn_Rate,Average_Risk,Average_ARPU
0,Broad Recent Deterioration Review,843,146,17.319098,0.155870,187.411028
1,Persistent Deterioration Review,3424,584,17.056075,0.156724,189.656903
2,General Retention Review,2997,295,9.843177,0.092885,344.212226
3,Recharge / Affordability Review,4627,349,7.542684,0.070698,297.860618
4,Usage / Engagement Review,2109,52,2.465624,0.022556,353.454827


In [19]:
# ============================================================
# Very High Risk Action Analysis
#
# Purpose:
# Examines candidate action categories specifically among
# customers already identified as Very High Risk.
#
# This helps determine whether behavioural routing provides
# useful context inside the highest model-risk population.
# ============================================================


very_high_action_summary = (
    retention_df[
        retention_df["risk_tier"] == "Very High"
    ]
    .groupby("candidate_action")
    .agg(
        # Number of Very High Risk customers assigned to the
        # candidate review category.
        Customers=("id", "count"),

        # Number of observed churners within the category.
        Actual_Churners=(TARGET, "sum"),

        # Observed churn rate for the Very High Risk subset.
        Churn_Rate=(TARGET, "mean"),

        # Average predicted churn probability.
        Average_Risk=("predicted_churn_risk", "mean"),

        # Average latest observed ARPU.
        Average_ARPU=(VALUE_FEATURE, "mean")
    )
    .reset_index()
)


# Convert churn rate into percentage form.
very_high_action_summary["Churn_Rate"] = (
    very_high_action_summary["Churn_Rate"] * 100
)


# Sort by observed churn rate for easier interpretation.
very_high_action_summary = (
    very_high_action_summary
    .sort_values(
        "Churn_Rate",
        ascending=False
    )
    .reset_index(drop=True)
)


very_high_action_summary

,candidate_action,Customers,Actual_Churners,Churn_Rate,Average_Risk,Average_ARPU
0,Persistent Deterioration Review,1298,548,42.218798,0.397810,99.720096
1,General Retention Review,681,274,40.234949,0.382144,223.672477
2,Broad Recent Deterioration Review,371,137,36.927224,0.338711,142.575927
3,Recharge / Affordability Review,927,307,33.117584,0.322862,196.559912
4,Usage / Engagement Review,223,36,16.143498,0.150174,352.119570


In [21]:
# ============================================================
# Final Retention Intelligence Dataset
#
# Purpose:
# Consolidates the finalized analytical outputs into a single
# customer-level decision dataset for downstream dashboard
# and AI-agent consumption.
#
# Only decision-relevant fields are retained. The full prepared
# dataset remains available separately for analytical/modeling
# reproducibility.
# ============================================================


# Select the customer identity, model risk, behavioural context,
# value information, prioritization and candidate action fields.
retention_intelligence_columns = [
    "id",

    # Model-derived churn risk.
    "predicted_churn_risk",
    "risk_tier",

    # Behavioural deterioration representation.
    "behavioral_state",
    "recent_deterioration_count",
    "persistent_deterioration_count",
    "coordinated_deterioration_count",

    # Customer-value representation.
    "value_tier",
    VALUE_FEATURE,

    # Operational retention decision layer.
    "retention_priority",
    "candidate_action",

    # Retained only for validation and offline evaluation.
    TARGET
]


# Create the final customer-level decision dataset.
retention_intelligence_df = (
    retention_df[
        retention_intelligence_columns
    ]
    .copy()
)


print("Retention intelligence dataset created.")
print(
    f"Customers : "
    f"{len(retention_intelligence_df):,}"
)
print(
    f"Columns   : "
    f"{len(retention_intelligence_df.columns)}"
)

Retention intelligence dataset created.
Customers : 14,000
Columns   : 12


In [22]:
# ============================================================
# Behavioural Evidence Construction
#
# Purpose:
# Converts the underlying deterioration indicators into a
# concise evidence string that can be displayed directly in
# the dashboard or supplied as context to the retention agent.
#
# The evidence describes observable data patterns only.
# It does not infer customer intent or guarantee a cause.
# ============================================================


def build_behavioral_evidence(row):
    # Collect the observable deterioration dimensions associated
    # with the customer's current behavioural classification.
    evidence = []

    if row["recent_deterioration_count"] > 0:
        evidence.append(
            f"{row['recent_deterioration_count']}/5 "
            "behavioural metrics declined recently"
        )

    if row["persistent_deterioration_count"] > 0:
        evidence.append(
            f"{row['persistent_deterioration_count']}/5 "
            "behavioural metrics show persistent decline"
        )

    if row["coordinated_deterioration_count"] > 0:
        evidence.append(
            f"{row['coordinated_deterioration_count']}/5 "
            "behavioural metrics declined across the observed period"
        )

    # Provide a neutral fallback when no deterioration signal
    # is present in the engineered behavioural representation.
    if not evidence:
        evidence.append(
            "No strong engineered deterioration signal detected"
        )

    return "; ".join(evidence)


retention_intelligence_df["behavioral_evidence"] = (
    retention_intelligence_df.apply(
        build_behavioral_evidence,
        axis=1
    )
)


print("Behavioural evidence generated successfully.")

Behavioural evidence generated successfully.


In [23]:
# ============================================================
# Decision Rationale Construction
#
# Purpose:
# Creates a compact explanation of why the customer received
# the assigned retention priority.
#
# This rationale is derived entirely from the frozen analytical
# layers and is intended to make downstream decisions auditable.
# ============================================================


def build_decision_rationale(row):
    # Priority 1 represents high-risk customers with relatively
    # high observed revenue exposure.
    if row["retention_priority"] == "Priority 1":
        return (
            "Very High churn risk with Higher/Highest relative "
            "customer value."
        )

    # Priority 2 represents very-high-risk customers with lower
    # or moderate value but strong observable deterioration.
    elif row["retention_priority"] == "Priority 2":
        return (
            "Very High churn risk combined with strong behavioural "
            "deterioration."
        )

    # Priority 3 contains all remaining customers.
    return (
        "Customer does not meet the higher retention-priority "
        "criteria."
    )


retention_intelligence_df["decision_rationale"] = (
    retention_intelligence_df.apply(
        build_decision_rationale,
        axis=1
    )
)


print("Decision rationales generated successfully.")

Decision rationales generated successfully.


In [24]:
# ============================================================
# Final Schema Ordering
#
# Purpose:
# Places the fields in the same logical sequence used by the
# retention decision process: identity → risk → behaviour →
# value → priority → action → explanation → evaluation.
# ============================================================


final_columns = [
    "id",

    "predicted_churn_risk",
    "risk_tier",

    "behavioral_state",
    "behavioral_evidence",
    "recent_deterioration_count",
    "persistent_deterioration_count",
    "coordinated_deterioration_count",

    "value_tier",
    VALUE_FEATURE,

    "retention_priority",
    "candidate_action",
    "decision_rationale",

    TARGET
]


retention_intelligence_df = (
    retention_intelligence_df[
        final_columns
    ]
    .copy()
)


print("Final schema established.")
print(
    retention_intelligence_df.columns.tolist()
)

Final schema established.
['id', 'predicted_churn_risk', 'risk_tier', 'behavioral_state', 'behavioral_evidence', 'recent_deterioration_count', 'persistent_deterioration_count', 'coordinated_deterioration_count', 'value_tier', 'arpu_8', 'retention_priority', 'candidate_action', 'decision_rationale', 'churn_probability']


In [25]:
retention_intelligence_df.head(10)

,id,predicted_churn_risk,risk_tier,behavioral_state,behavioral_evidence,recent_deterioration_count,persistent_deterioration_count,coordinated_deterioration_count,value_tier,arpu_8,retention_priority,candidate_action,decision_rationale,churn_probability
25294,25294,0.035815,Very High,Stable / Limited Deterioration,1/5 behavioural metrics declined recently; 1/5...,1,1,3,Lower,35.192,Priority 3,Usage / Engagement Review,Customer does not meet the higher retention-pr...,0
24327,24327,0.597727,Very High,Persistent Deterioration,4/5 behavioural metrics declined recently; 2/5...,4,2,5,Lower,0.970,Priority 2,Persistent Deterioration Review,Very High churn risk combined with strong beha...,0
58676,58676,0.009151,High,Stable / Limited Deterioration,1/5 behavioural metrics declined recently; 1/5...,1,1,4,Moderate,109.958,Priority 3,Recharge / Affordability Review,Customer does not meet the higher retention-pr...,0
32271,32271,0.002134,Low,Stable / Limited Deterioration,1/5 behavioural metrics declined recently,1,0,0,Highest,435.348,Priority 3,Recharge / Affordability Review,Customer does not meet the higher retention-pr...,0
25682,25682,0.149513,Very High,Persistent Deterioration,2/5 behavioural metrics declined recently; 2/5...,2,2,4,Highest,411.319,Priority 1,Persistent Deterioration Review,Very High churn risk with Higher/Highest relat...,0
43551,43551,0.005331,Moderate,Recent Deterioration,2/5 behavioural metrics declined recently,2,0,0,Moderate,97.050,Priority 3,Recharge / Affordability Review,Customer does not meet the higher retention-pr...,0
24894,24894,0.009031,High,Stable / Limited Deterioration,1/5 behavioural metrics declined across the ob...,0,0,1,Moderate,101.323,Priority 3,General Retention Review,Customer does not meet the higher retention-pr...,0
30098,30098,0.019629,High,Severe Persistent Deterioration,5/5 behavioural metrics declined recently; 4/5...,5,4,4,Moderate,86.426,Priority 3,Persistent Deterioration Review,Customer does not meet the higher retention-pr...,0
30086,30086,0.001152,Low,Stable / Limited Deterioration,No strong engineered deterioration signal dete...,0,0,0,Moderate,153.054,Priority 3,General Retention Review,Customer does not meet the higher retention-pr...,0
38248,38248,0.004489,Moderate,Recent Broad Deterioration,4/5 behavioural metrics declined recently; 4/5...,4,0,4,Moderate,182.124,Priority 3,Broad Recent Deterioration Review,Customer does not meet the higher retention-pr...,0


In [27]:
# ============================================================
# Retention Intelligence Consistency Check
#
# Purpose:
# Verifies that each customer's categorical decision fields
# remain consistent with the underlying numerical values and
# the frozen segmentation logic.
#
# This check prevents a downstream dashboard or AI agent from
# consuming internally inconsistent customer records.
# ============================================================


# Display the risk distribution and corresponding score ranges.
risk_validation = (
    retention_intelligence_df
    .groupby("risk_tier", observed=False)
    .agg(
        Customers=("id", "count"),
        Minimum_Risk=("predicted_churn_risk", "min"),
        Maximum_Risk=("predicted_churn_risk", "max"),
        Average_Risk=("predicted_churn_risk", "mean")
    )
    .reset_index()
)

risk_validation

,risk_tier,Customers,Minimum_Risk,Maximum_Risk,Average_Risk
0,Low,3500,0.000096,0.003264,0.001785
1,Moderate,3500,0.003265,0.008827,0.005639
2,High,3500,0.008831,0.035595,0.017160
3,Very High,3500,0.035595,0.991963,0.352869


In [28]:
# ============================================================
# Risk Tier Consistency Check
#
# Purpose:
# Identifies records where the assigned risk tier appears
# inconsistent with the customer's predicted churn probability.
# ============================================================


risk_validation_errors = retention_intelligence_df[
    (
        (retention_intelligence_df["risk_tier"] == "Low")
        & (
            retention_intelligence_df["predicted_churn_risk"]
            > retention_intelligence_df[
                "predicted_churn_risk"
            ].quantile(0.25)
        )
    )
    |
    (
        (retention_intelligence_df["risk_tier"] == "Very High")
        & (
            retention_intelligence_df["predicted_churn_risk"]
            < retention_intelligence_df[
                "predicted_churn_risk"
            ].quantile(0.75)
        )
    )
]


print(
    f"Potential risk-tier inconsistencies : "
    f"{len(risk_validation_errors):,}"
)

risk_validation_errors.head(20)

Potential risk-tier inconsistencies : 0


,id,predicted_churn_risk,risk_tier,behavioral_state,behavioral_evidence,recent_deterioration_count,persistent_deterioration_count,coordinated_deterioration_count,value_tier,arpu_8,retention_priority,candidate_action,decision_rationale,churn_probability


In [29]:
# ============================================================
# Persist Final Retention Intelligence Dataset
#
# Purpose:
# Saves the validated customer-level decision layer as the
# official downstream input for the retention dashboard and
# AI retention agent.
# ============================================================


RETENTION_OUTPUT_PATH = (
    OUTPUT_DIR
    / "telecom_retention_intelligence.csv"
)


retention_intelligence_df.to_csv(
    RETENTION_OUTPUT_PATH,
    index=False
)


print("Final retention intelligence dataset saved.")
print(
    f"Path : {RETENTION_OUTPUT_PATH.resolve()}"
)
print(
    f"Rows : {len(retention_intelligence_df):,}"
)
print(
    f"Cols : {len(retention_intelligence_df.columns)}"
)

Final retention intelligence dataset saved.
Path : E:\Churn Prediction\data\processed\telecom_retention_intelligence.csv
Rows : 14,000
Cols : 14
